# 로사리움 - 성모상 3D 모델 만들기 (사진 6장 → GLB)

여러 각도에서 본 **같은 조각상 사진 1~6장**으로 텍스처가 입혀진 3D 모델(GLB)을 만듭니다.
무료 Colab(T4 GPU)에서 동작하도록 만든 노트북입니다. 코드를 읽거나 고칠 필요는 없습니다.

## 사용 방법 (4단계)
1. 위 메뉴 **런타임 → 런타임 유형 변경** → 하드웨어 가속기 **T4 GPU** 선택 → **저장**
2. 위 메뉴 **런타임 → 모두 실행** (단축키 `Ctrl + F9`)
3. 잠시 후 아래 **「2. 사진 올리기」** 칸에 **파일 선택** 버튼이 나타나면, 사진 6장을 한꺼번에 선택해서 올립니다.
   (jpg / png / webp, 또는 사진들을 묶은 zip 1개. 파일 이름에 `front`가 들어간 사진을 정면으로 사용합니다.)
4. 기다리면 끝에 GLB 파일 2개가 **자동으로 다운로드**됩니다.
   - `mary_statue_trellis.glb` : 기본 품질 (삼각형 약 2만~5만 개)
   - `mary_statue_trellis_mobile.glb` : 모바일 앱용 경량판 (삼각형 약 1만5천 개, 텍스처 1024)

## 걸리는 시간 (무료 T4 기준, 대략)
| 단계 | 시간 |
|---|---|
| 프로그램 설치 (처음 1회) | 10~15분 |
| 모델 다운로드 (처음 1회, 약 4.5GB) | 2~5분 |
| 3D 생성 + 미리보기 | 3~6분 |
| GLB 2종 만들기 | 5~10분 |
| **합계** | **약 20~35분** |

사진을 올린 뒤에는 창을 열어 둔 채 기다리기만 하면 됩니다. (브라우저 탭을 닫으면 작업이 멈출 수 있습니다.)

## 문제가 생기면
- **"GPU가 없습니다"** → 1번 단계(T4 GPU 선택)를 하고 다시 **모두 실행**.
- **"GPU 사용량 한도에 도달했습니다" / 연결할 수 없음** → 무료 Colab의 하루 사용량 한도입니다. 몇 시간~하루 뒤에 다시 시도하세요.
- **설치 단계에서 빨간 오류** → **런타임 → 세션 다시 시작** 후 **모두 실행**. 그래도 안 되면 **런타임 → 런타임 연결 해제 및 삭제** 후 처음부터.
- **"GPU 메모리 부족"** → 노트북이 자동으로 사진 3장으로 다시 시도하고, 그래도 안 되면 대체 방법(TripoSR)으로 만듭니다.
- 자세한 내용은 맨 아래 **「문제 해결」** 을 보세요.

> 사용 모델: Microsoft **TRELLIS** (`microsoft/TRELLIS-image-large`, MIT 라이선스), 대체용 **TripoSR** (MIT). 로그인/토큰/유료 서비스 없이 동작합니다.


## 0. 설정
보통은 그대로 두면 됩니다. 결과가 마음에 들지 않으면 `SEED`(무작위 시드)만 바꿔서 **3번부터 다시 실행**해 보세요.


In [ ]:
# @title 설정 (보통은 바꾸지 않아도 됩니다)
SEED = 1  # @param {type:"integer"}
# 생성 단계 수. 6장의 사진이 고르게 쓰이도록 6의 배수를 권장 (12 = 빠름, 24 = 기본, 30 = 조금 더 정교)
STEPS = 24  # @param {type:"integer"}
SS_GUIDANCE = 7.5  # @param {type:"number"}
SLAT_GUIDANCE = 3.0  # @param {type:"number"}
# 여러 장 결합 방식: stochastic(기본, 형태 왜곡이 적음) / multidiffusion(모든 사진 평균, 느림)
MULTI_MODE = "stochastic"  # @param ["stochastic", "multidiffusion"]
TEXTURE_SIZE = 1024  # @param [1024, 2048] {type:"raw"}
# 메시 단순화 비율 (0.95 = 삼각형 95% 제거 → 보통 2만~5만 개)
SIMPLIFY = 0.95  # @param {type:"number"}
MOBILE_TARGET_FACES = 15000  # @param {type:"integer"}
MOBILE_TEXTURE_SIZE = 1024  # @param [512, 1024] {type:"raw"}
# 배경 제거 모델: u2net(기본, 빠름) / birefnet-general-lite(더 정교, CPU에서 장당 약 1분)
REMBG_MODEL = "u2net"  # @param ["u2net", "birefnet-general-lite"]
# 파일 이름에 이 단어가 들어간 사진을 정면(첫 번째)으로 사용
FRONT_KEYWORD = "front"  # @param {type:"string"}
AUTO_DOWNLOAD = True  # @param {type:"boolean"}
# True 로 하면 TRELLIS 를 건너뛰고 대체 방법(TripoSR, 정면 1장)만 실행
FORCE_TRIPOSR = False  # @param {type:"boolean"}

OUTPUT_BASENAME = "mary_statue"

import os
ROOT = "/content/rosarium"
ENV_DIR = f"{ROOT}/env"
VENV_PY = f"{ENV_DIR}/bin/python"
SRC_DIR = f"{ROOT}/src"
WORK_DIR = f"{ROOT}/work"
OUT_DIR = f"{ROOT}/output"
INPUT_DIR = f"{ROOT}/inputs"
CACHE_DIR = f"{ROOT}/cache"
SCRIPT_DIR = f"{ROOT}/scripts"
LOG_DIR = f"{ROOT}/logs"
for _d in [ROOT, SRC_DIR, WORK_DIR, OUT_DIR, INPUT_DIR, CACHE_DIR, SCRIPT_DIR, LOG_DIR]:
    os.makedirs(_d, exist_ok=True)

# 고정 버전 (재현성)
TRELLIS_MODEL_REV = "25e0d31ffbebe4b5a97464dd851910efc3002d96"   # microsoft/TRELLIS-image-large
TRIPOSR_MODEL_REV = "5b521936b01fbe1890f6f9baed0254ab6351c04a"   # stabilityai/TripoSR
print("설정 완료. SEED =", SEED, "/ STEPS =", STEPS, "/ MODE =", MULTI_MODE)


## 1. GPU 확인


In [ ]:
# @title GPU 확인 (GPU 가 없으면 여기서 멈춥니다)
import os, sys, subprocess, shutil, json, time


class 중단(Exception):
    """노트북을 여기서 멈추기 위한 오류 (모두 실행 시 다음 칸으로 넘어가지 않음)."""


def clean_env(extra=None):
    """Colab 시스템 파이썬 설정이 별도 Python 3.10 환경에 섞이지 않도록 정리한 환경 변수."""
    env = dict(os.environ)
    for k in ["PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "MPLBACKEND", "VIRTUAL_ENV"]:
        env.pop(k, None)
    ld = [p for p in env.get("LD_LIBRARY_PATH", "").split(":") if p and "/usr/local/cuda" not in p]
    env["LD_LIBRARY_PATH"] = ":".join(ld)
    env.update({
        "PYTHONNOUSERSITE": "1",
        "PYTHONUNBUFFERED": "1",
        "HF_HOME": f"{CACHE_DIR}/hf",
        "HF_HUB_DISABLE_TELEMETRY": "1",
        "TORCH_HOME": f"{CACHE_DIR}/torch",
        "U2NET_HOME": f"{CACHE_DIR}/u2net",
        "UV_CACHE_DIR": f"{CACHE_DIR}/uv",
        "UV_PYTHON_INSTALL_DIR": f"{ROOT}/python",
        "UV_LINK_MODE": "copy",
        "PIP_DISABLE_PIP_VERSION_CHECK": "1",
        "DEBIAN_FRONTEND": "noninteractive",
        "ATTN_BACKEND": "xformers",
        "SPARSE_ATTN_BACKEND": "xformers",
        "SPCONV_ALGO": "native",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        "DINOV2_REPO": f"{SRC_DIR}/dinov2",
    })
    if extra:
        env.update(extra)
    return env


def run_stream(cmd, log_name, env=None, cwd=None):
    """명령을 실행하면서 출력을 실시간으로 보여 주고 로그 파일에도 남긴다. 종료 코드를 돌려준다."""
    log_path = os.path.join(LOG_DIR, log_name)
    with open(log_path, "ab") as log:
        log.write(("\n===== %s : %s\n" % (time.strftime("%H:%M:%S"), " ".join(cmd))).encode())
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             env=env or clean_env(), cwd=cwd)
        while True:
            chunk = p.stdout.read1(4096)
            if not chunk:
                break
            log.write(chunk)
            sys.stdout.write(chunk.decode("utf-8", "replace"))
            sys.stdout.flush()
        return p.wait()


def tail(log_name, n=40):
    try:
        lines = open(os.path.join(LOG_DIR, log_name), encoding="utf-8", errors="replace").read().splitlines()
        print("\n".join(lines[-n:]))
    except Exception:
        pass


_q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                    capture_output=True, text=True) if shutil.which("nvidia-smi") else None
if _q is None or _q.returncode != 0 or not _q.stdout.strip():
    print("=" * 70)
    print("GPU 가 연결되어 있지 않습니다.")
    print("  1) 메뉴 [런타임] → [런타임 유형 변경] → 하드웨어 가속기 'T4 GPU' → [저장]")
    print("  2) 메뉴 [런타임] → [모두 실행]")
    print("  'GPU 를 사용할 수 없습니다 / 사용량 한도' 안내가 나오면 무료 사용량을 다 쓴 것입니다.")
    print("  몇 시간~하루 뒤에 다시 시도해 주세요.")
    print("=" * 70)
    raise 중단("GPU 없음: T4 GPU 런타임으로 바꾼 뒤 다시 '모두 실행' 하세요.")

_name, _mem = [x.strip() for x in _q.stdout.strip().splitlines()[0].rsplit(",", 1)]
GPU_NAME, GPU_MEM_GB = _name, float(_mem) / 1024.0
try:  # compute capability (T4 = 7.5). 오래된 드라이버는 이 항목이 없어 기본값 사용
    _cc = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip().splitlines()[0]
    GPU_CC = "%.1f" % float(_cc)
except Exception:
    GPU_CC = "7.5"
print(f"GPU: {GPU_NAME}  메모리 {GPU_MEM_GB:.1f} GB  (compute capability {GPU_CC})")
if GPU_MEM_GB < 14:
    print("[주의] GPU 메모리가 14GB 보다 작습니다. 메모리 부족이 날 수 있습니다.")
if "T4" not in GPU_NAME:
    print("(T4 가 아닌 GPU 입니다. 더 좋은 GPU 라면 그대로 진행해도 됩니다.)")
_free = shutil.disk_usage("/content").free / 1e9
print(f"디스크 여유 공간: {_free:.0f} GB")
if _free < 20:
    print("[주의] 디스크 여유 공간이 20GB 보다 적습니다. 설치가 실패할 수 있습니다.")
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("[주의] Google Colab 이 아닌 곳에서 실행 중입니다. 업로드/다운로드 칸이 동작하지 않을 수 있습니다.")
json.dump({"name": GPU_NAME, "mem_gb": GPU_MEM_GB, "cc": GPU_CC}, open(f"{ROOT}/gpu.json", "w"))
print("GPU 확인 완료.")


## 2. 사진 올리기
아래 칸에 **파일 선택** 버튼이 나타나면 사진을 **한꺼번에** 고르세요 (Ctrl 또는 Shift 를 누른 채 여러 개 선택).
- 1~6장, jpg/png/webp, 또는 zip 1개
- 흰 배경의 같은 조각상, 정면 / 비스듬한 앞 / 옆 / 뒤 등 여러 각도
- 사진을 올린 다음에는 기다리기만 하면 됩니다. 설치와 생성은 자동으로 이어서 진행됩니다.


In [ ]:
# @title 사진 올리기 + 미리보기
import io, re, zipfile
from PIL import Image, ImageOps
from IPython.display import display

EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}


def _safe(name):
    base, ext = os.path.splitext(os.path.basename(name))
    base = re.sub(r"[^0-9A-Za-z가-힣_.-]+", "_", base).strip("._") or "image"
    return base + ext.lower()


def _natural_key(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", os.path.basename(p))]


def _save(name, data, saved):
    if os.path.splitext(name)[1].lower() not in EXTS:
        return
    out = os.path.join(INPUT_DIR, _safe(name))
    k = 1
    while os.path.exists(out):
        b, e = os.path.splitext(_safe(name))
        out = os.path.join(INPUT_DIR, f"{b}_{k}{e}")
        k += 1
    with open(out, "wb") as f:
        f.write(data)
    try:
        with Image.open(out) as im:
            im.verify()
        saved.append(out)
    except Exception:
        os.remove(out)
        print(f"  [건너뜀] 이미지로 읽을 수 없는 파일: {name}")


if IN_COLAB:
    from google.colab import files
    shutil.rmtree(INPUT_DIR, ignore_errors=True)
    os.makedirs(INPUT_DIR, exist_ok=True)
    print("[파일 선택] 버튼을 눌러 사진 1~6장(또는 zip 1개)을 한꺼번에 고르세요.")
    uploaded = files.upload()
    saved = []
    for name, data in uploaded.items():
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                for info in z.infolist():
                    n = os.path.basename(info.filename)
                    if info.is_dir() or not n or n.startswith(".") or "__MACOSX" in info.filename:
                        continue
                    _save(n, z.read(info), saved)
        else:
            _save(name, data, saved)
else:
    saved = [os.path.join(INPUT_DIR, f) for f in os.listdir(INPUT_DIR)]

saved = sorted(saved, key=_natural_key)
if not saved:
    raise 중단("올린 이미지가 없습니다. 이 칸을 다시 실행(▶)해서 사진을 올린 뒤, 메뉴 [런타임] → [이후 셀 실행] 을 누르세요.")

# 정면 사진을 맨 앞으로 (TRELLIS stochastic 모드는 첫 단계에 첫 번째 사진을 씀)
kw = (FRONT_KEYWORD or "front").lower()
_bad = ("quarter", "three", "3q", "back", "side")
_front = [p for p in saved if kw in os.path.basename(p).lower()
          and not any(b in os.path.basename(p).lower() for b in _bad)]
if _front:
    saved.remove(_front[0])
    saved.insert(0, _front[0])
if len(saved) > 6:
    print(f"[주의] 사진이 {len(saved)}장입니다. 앞의 6장만 사용합니다.")
    saved = saved[:6]
INPUT_IMAGES = saved
json.dump(INPUT_IMAGES, open(f"{ROOT}/inputs.json", "w"), ensure_ascii=False, indent=2)

print(f"\n사용할 사진 {len(INPUT_IMAGES)}장 (1번 = 정면):")
thumbs = []
for i, p in enumerate(INPUT_IMAGES):
    with Image.open(p) as im:
        im = ImageOps.exif_transpose(im)
        print(f"  {i + 1}. {os.path.basename(p)}  ({im.width}x{im.height})")
        t = im.convert("RGB")
        t.thumbnail((220, 300))
        thumbs.append(t)
sheet = Image.new("RGB", (230 * len(thumbs), 310), (255, 255, 255))
for i, t in enumerate(thumbs):
    sheet.paste(t, (230 * i + (230 - t.width) // 2, (310 - t.height) // 2))
display(sheet)


## 3. 프로그램 설치 (처음 1회, 10~15분)
Colab 기본 파이썬과 섞이지 않도록 **별도의 Python 3.10 환경**에 TRELLIS 가 검증된 버전
(PyTorch 2.4.0 + CUDA 12.1, xformers 0.0.27.post2, spconv-cu120 2.3.6 등)을 설치하고,
T4 용으로 렌더러 2개(nvdiffrast, diff-gaussian-rasterization)를 직접 컴파일합니다.
다시 실행해도 이미 끝난 단계는 건너뜁니다.


In [ ]:
# @title 설치 (처음 1회 10~15분, 다시 실행하면 끝난 단계는 건너뜀)
INSTALL_BASE = r"""#!/usr/bin/env bash
# 기본 환경: Python 3.10 + PyTorch 2.4.0(cu121) + TRELLIS 의존성 (TripoSR 대체 경로도 이 환경 사용)
set -eo pipefail
ROOT=/content/rosarium
ENV=$ROOT/env
PY=$ENV/bin/python
UV="${UV_BIN:-uv}"
if [ -f "$ROOT/.ok_base" ] && "$PY" -c "import torch, xformers, spconv.pytorch, rembg, trimesh" 2>/dev/null; then
  echo "[설치] 기본 환경: 이미 설치됨 (건너뜀)"; exit 0
fi
if [ ! -x "$PY" ]; then
  echo "[설치] Python 3.10 환경 만드는 중..."
  "$UV" venv --managed-python --python 3.10 --seed "$ENV"
fi
"$PY" -c "import sys; assert sys.version_info[:2] == (3, 10), sys.version"
echo "[설치] PyTorch 2.4.0 (CUDA 12.1) 와 의존성 설치 중... (약 3~5분, 2.5GB 다운로드)"
"$UV" pip install --python "$PY" --index-strategy unsafe-best-match \
  --extra-index-url https://download.pytorch.org/whl/cu121 \
  "torch==2.4.0" "torchvision==0.19.0" "numpy==1.26.4" "xformers==0.0.27.post2" \
  "spconv-cu120==2.3.6" "pillow==10.4.0" "imageio==2.36.1" "imageio-ffmpeg==0.5.1" \
  "tqdm==4.67.1" "easydict==1.13" "opencv-python-headless==4.10.0.84" "scipy==1.14.1" \
  "rembg==2.0.60" "onnxruntime==1.20.1" "trimesh==4.5.3" "xatlas==0.0.9" "pyvista==0.44.2" \
  "pymeshfix==0.17.0" "igraph==0.11.8" "plyfile==1.1.3" "huggingface_hub==0.26.5" \
  "safetensors==0.4.5" "setuptools==84.2.1" "wheel==0.48.0" "ninja==1.13.2" \
  "utils3d @ git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8"
"$PY" - <<'PYEOF'
import torch, torchvision, xformers, spconv, numpy
print("  torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU 사용 가능:", torch.cuda.is_available())
print("  xformers", xformers.__version__, "| numpy", numpy.__version__)
assert torch.cuda.is_available(), "PyTorch 가 GPU 를 찾지 못했습니다"
PYEOF
touch "$ROOT/.ok_base"
echo "[설치] 기본 환경 완료"
"""

INSTALL_TRELLIS = r"""#!/usr/bin/env bash
# TRELLIS 전용: CUDA 12.4 컴파일러(NVIDIA 공식 redist, sha256 확인) + 소스(커밋 고정) + T4 용 확장 컴파일
set -eo pipefail
ROOT=/content/rosarium
ENV=$ROOT/env
PY=$ENV/bin/python
SRC=$ROOT/src
CUDA=$ROOT/cuda-12.4
UV="${UV_BIN:-uv}"
ARCH="${GPU_ARCH:-7.5}"
mkdir -p "$SRC" "$ROOT/dl"

# ---- 1) nvcc 12.4 (OS 와 무관하게 동작하도록 NVIDIA redist tar 를 직접 받음)
if [ ! -f "$CUDA/.ok" ]; then
  echo "[설치] CUDA 12.4 컴파일러 받는 중 (약 55MB)..."
  rm -rf "$CUDA"; mkdir -p "$CUDA"
  B=https://developer.download.nvidia.com/compute/cuda/redist
  while read -r rel sha; do
    [ -z "$rel" ] && continue
    f="$ROOT/dl/$(basename "$rel")"
    if ! echo "$sha  $f" | sha256sum -c --quiet - 2>/dev/null; then
      curl -fsSL --retry 3 -o "$f" "$B/$rel"
      echo "$sha  $f" | sha256sum -c --quiet -
    fi
    tar -xJf "$f" -C "$CUDA" --strip-components=1
  done <<'EOF'
cuda_nvcc/linux-x86_64/cuda_nvcc-linux-x86_64-12.4.131-archive.tar.xz 7ffba1ada0e4b8c17e451ac7a60d386aa2642ecd08d71202a0b100c98bd74681
cuda_cudart/linux-x86_64/cuda_cudart-linux-x86_64-12.4.127-archive.tar.xz 0483bff9a36e7a44465db3cd42874f6f70f019297dcf803fbefcbf58d7448c8f
cuda_cccl/linux-x86_64/cuda_cccl-linux-x86_64-12.4.127-archive.tar.xz e1636f27a142d24e73dfd831c54bbf5575b498fd5900648d7372fae46f824fdf
cuda_nvtx/linux-x86_64/cuda_nvtx-linux-x86_64-12.4.127-archive.tar.xz f3afccda5ff1e7521cca74153e92708349148294bc75559afe64d57c43763454
cuda_profiler_api/linux-x86_64/cuda_profiler_api-linux-x86_64-12.4.127-archive.tar.xz fa11c08e39ab35453e07dd5012adc0767408955431fc4acdf64ca4cdd7692646
EOF
  # cuBLAS/cuSPARSE/cuSOLVER/cuRAND 헤더는 PyTorch 가 설치한 nvidia-* 패키지에서 복사 (PyTorch 헤더가 include 함)
  SP=$("$PY" -c "import site; print(site.getsitepackages()[0])")/nvidia
  for p in cublas cusparse cusolver curand; do cp -rn "$SP/$p/include/." "$CUDA/include/" 2>/dev/null || cp -r "$SP/$p/include/." "$CUDA/include/"; done
  rm -f "$CUDA/include/__init__.py"
  ln -sfn lib "$CUDA/lib64"
  "$CUDA/bin/nvcc" --version | tail -n 2
  touch "$CUDA/.ok"
fi

# ---- 2) 소스 코드 (커밋 고정, 얕은 다운로드)
fetch_at() {
  local url=$1 dir=$2 rev=$3
  if [ "$(git -C "$dir" rev-parse HEAD 2>/dev/null)" = "$rev" ]; then return 0; fi
  rm -rf "$dir"; mkdir -p "$dir"
  git -C "$dir" init -q
  git -C "$dir" remote add origin "$url"
  git -C "$dir" fetch -q --depth 1 origin "$rev"
  git -C "$dir" -c advice.detachedHead=false checkout -q FETCH_HEAD
}
echo "[설치] 소스 코드 받는 중..."
fetch_at https://github.com/microsoft/TRELLIS.git "$SRC/TRELLIS" 193e0502b5cb3eb5997ec457d5340e95d09b158e
FC="$SRC/TRELLIS/trellis/representations/mesh/flexicubes"
if [ ! -f "$FC/flexicubes.py" ]; then
  fetch_at https://github.com/MaxtirError/FlexiCubes.git "$FC" 815e075a2a400d06c48d94c347674344ed6ae5c5
fi
fetch_at https://github.com/facebookresearch/dinov2.git "$SRC/dinov2" 7764ea0f912e53c92e82eb78a2a1631e92725fc8
fetch_at https://github.com/autonomousvision/mip-splatting.git "$SRC/mip-splatting" dda02ab5ecf45d6edb8c540d9bb65c7e451345a9
fetch_at https://github.com/NVlabs/nvdiffrast.git "$SRC/nvdiffrast" 253ac4fcea7de5f396371124af597e6cc957bfae

# ---- 3) TRELLIS 작은 패치 (반복 실행해도 안전)
"$PY" - "$SRC/TRELLIS" <<'PYEOF'
import os, sys, site
root = sys.argv[1]
# (a) text-to-3D 파이프라인은 쓰지 않음: open3d/transformers 없이도 import 되도록
p = os.path.join(root, "trellis/pipelines/__init__.py")
s = open(p).read()
old = "from .trellis_text_to_3d import TrellisTextTo3DPipeline\n"
if "TrellisTextTo3DPipeline = None" not in s:
    assert s.count(old) == 1, "TRELLIS 패치 실패 (pipelines/__init__.py)"
    s = s.replace(old, "try:\n    from .trellis_text_to_3d import TrellisTextTo3DPipeline\nexcept ImportError:\n    TrellisTextTo3DPipeline = None\n")
    open(p, "w").write(s)
# (b) DINOv2 를 GitHub API 대신 고정된 로컬 사본에서 불러오기 (공유 IP 의 GitHub API 제한 회피)
p = os.path.join(root, "trellis/pipelines/trellis_image_to_3d.py")
s = open(p).read()
old = "torch.hub.load('facebookresearch/dinov2', name, pretrained=True)"
new = ("(torch.hub.load(__import__('os').environ['DINOV2_REPO'], name, source='local', pretrained=True) "
       "if __import__('os').environ.get('DINOV2_REPO') else torch.hub.load('facebookresearch/dinov2', name, pretrained=True))")
if "DINOV2_REPO" not in s:
    assert s.count(old) == 1, "TRELLIS 패치 실패 (dinov2)"
    s = s.replace(old, new)
    open(p, "w").write(s)
assert "DINOV2_REPO" in open(p).read(), "TRELLIS 패치 실패 (dinov2)"
# (c) FlexiCubes 는 kaolin 의 check_tensor 하나만 씀 -> 무거운 kaolin 대신 같은 동작의 작은 모듈
try:
    from kaolin.utils.testing import check_tensor  # noqa
except Exception:
    k = os.path.join(site.getsitepackages()[0], "kaolin")
    os.makedirs(os.path.join(k, "utils"), exist_ok=True)
    open(os.path.join(k, "__init__.py"), "w").write("# minimal shim for TRELLIS (only kaolin.utils.testing.check_tensor)\n")
    open(os.path.join(k, "utils", "__init__.py"), "w").write("")
    open(os.path.join(k, "utils", "testing.py"), "w").write('''def check_tensor(tensor, shape=None, dtype=None, device=None, throw=True):
    if shape is not None:
        if len(shape) != tensor.ndim:
            if throw:
                raise ValueError("tensor has %d dims, expected %d" % (tensor.ndim, len(shape)))
            return False
        for i, dim in enumerate(shape):
            if dim is not None and tensor.shape[i] != dim:
                if throw:
                    raise ValueError("tensor shape %s != %s" % (tuple(tensor.shape), shape))
                return False
    if dtype is not None and dtype != tensor.dtype:
        if throw:
            raise TypeError("tensor dtype %s != %s" % (tensor.dtype, dtype))
        return False
    if device is not None and str(device) != str(tensor.device.type):
        if throw:
            raise TypeError("tensor device %s != %s" % (tensor.device, device))
        return False
    return True
''')
print("  TRELLIS 패치 완료")
PYEOF

# ---- 4) 이 GPU(compute capability $ARCH) 전용으로 CUDA 확장 컴파일
export CUDA_HOME="$CUDA" PATH="$CUDA/bin:$ENV/bin:$PATH" TORCH_CUDA_ARCH_LIST="$ARCH" MAX_JOBS="$(nproc)"
GCC_MAJOR=$(gcc -dumpversion | cut -d. -f1)
if [ "$GCC_MAJOR" -gt 13 ]; then export NVCC_APPEND_FLAGS="-allow-unsupported-compiler"; fi
BUILT_ARCH=$(cat "$ROOT/.built_arch" 2>/dev/null || true)
if [ "$BUILT_ARCH" != "$ARCH" ] || ! "$PY" -c "import diff_gaussian_rasterization" 2>/dev/null; then
  echo "[설치] diff-gaussian-rasterization 컴파일 중 (sm_${ARCH/./}, 약 2~4분)..."
  "$UV" pip install --python "$PY" --no-build-isolation --reinstall-package diff_gaussian_rasterization \
    "$SRC/mip-splatting/submodules/diff-gaussian-rasterization"
fi
if [ "$BUILT_ARCH" != "$ARCH" ] || ! "$PY" -c "import nvdiffrast.torch" 2>/dev/null; then
  echo "[설치] nvdiffrast 0.4.0 컴파일 중 (약 4~8분)..."
  "$UV" pip install --python "$PY" --no-build-isolation --reinstall-package nvdiffrast "$SRC/nvdiffrast"
fi
echo "$ARCH" > "$ROOT/.built_arch"

# ---- 5) GPU 동작 시험 (작은 연산으로 각 확장이 이 GPU 에서 실제로 도는지 확인)
cd "$SRC/TRELLIS"
"$PY" - <<'PYEOF'
import torch
dev = "cuda"
import nvdiffrast.torch as dr
ctx = dr.RasterizeCudaContext()
pos = torch.tensor([[[-0.8, -0.8, 0, 1], [0.8, -0.8, 0, 1], [0, 0.8, 0, 1]]], device=dev, dtype=torch.float32)
tri = torch.tensor([[0, 1, 2]], device=dev, dtype=torch.int32)
rast, _ = dr.rasterize(ctx, pos, tri, resolution=[64, 64])
assert rast[..., 3].max() > 0, "nvdiffrast 시험 실패"
print("  nvdiffrast OK")
import xformers.ops as xops
q = torch.randn(1, 256, 8, 64, device=dev, dtype=torch.float16)
xops.memory_efficient_attention(q, q, q)
print("  xformers OK")
import spconv.pytorch as spconv
idx = torch.randperm(512, device=dev)[:64]
coords = torch.stack([torch.zeros_like(idx), idx // 64, (idx // 8) % 8, idx % 8], 1).int()
for dt in (torch.float32, torch.float16):
    x = spconv.SparseConvTensor(torch.randn(64, 16, device=dev, dtype=dt), coords, [8, 8, 8], 1)
    conv = spconv.SubMConv3d(16, 16, 3, algo=spconv.ConvAlgo.Native, indice_key="t").to(dev, dt)
    conv(x)
print("  spconv OK")
from diff_gaussian_rasterization import GaussianRasterizer, GaussianRasterizationSettings  # noqa
from trellis.pipelines import TrellisImageTo3DPipeline  # noqa
from trellis.utils import render_utils, postprocessing_utils  # noqa
print("  TRELLIS import OK")
PYEOF
touch "$ROOT/.ok_trellis"
echo "[설치] TRELLIS 준비 완료"
"""

open(f"{SCRIPT_DIR}/install_base.sh", "w").write(INSTALL_BASE)
open(f"{SCRIPT_DIR}/install_trellis.sh", "w").write(INSTALL_TRELLIS)

# uv: 빠른 파이썬 패키지 설치 도구 (버전 고정). Colab 에 다른 버전이 있어도 이 버전의 실행 파일을 직접 사용
def find_uv():
    r = subprocess.run([sys.executable, "-c", "import uv, sys; sys.stdout.write(uv.find_uv_bin())"],
                       capture_output=True, text=True)
    if r.returncode == 0 and os.path.exists(r.stdout.strip()):
        v = subprocess.run([r.stdout.strip(), "--version"], capture_output=True, text=True).stdout
        if "0.11.32" in v:
            return r.stdout.strip()
    return None


_uv = find_uv()
if not _uv:
    print("[설치] uv 0.11.32 설치 중...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.11.32"], check=True)
    _uv = find_uv()
    if not _uv:
        raise 중단("uv 설치 실패. [런타임] → [세션 다시 시작] 후 [모두 실행] 하세요.")
UV_BIN = _uv
_env = clean_env({"UV_BIN": _uv, "GPU_ARCH": globals().get("GPU_CC", "7.5")})

t0 = time.time()
rc = run_stream(["bash", f"{SCRIPT_DIR}/install_base.sh"], "install_base.log", env=_env)
if rc != 0:
    print("\n" + "=" * 70)
    print("기본 설치에 실패했습니다. (인터넷/서버 일시 오류일 수 있습니다)")
    print("  메뉴 [런타임] → [세션 다시 시작] 후 [모두 실행] 을 해 보세요.")
    print("  계속 실패하면 [런타임] → [런타임 연결 해제 및 삭제] 후 처음부터 다시.")
    print("=" * 70)
    raise 중단("기본 설치 실패 (로그: /content/rosarium/logs/install_base.log)")

TRELLIS_READY = False
if FORCE_TRIPOSR:
    print("FORCE_TRIPOSR = True 이므로 TRELLIS 설치를 건너뜁니다.")
else:
    rc = run_stream(["bash", f"{SCRIPT_DIR}/install_trellis.sh"], "install_trellis.log", env=_env)
    TRELLIS_READY = (rc == 0)
    if not TRELLIS_READY:
        print("\n" + "=" * 70)
        print("TRELLIS 설치(컴파일)에 실패했습니다. 아래 대체 방법(TripoSR)으로 자동 진행합니다.")
        print("TRELLIS 로 다시 시도하려면: [런타임] → [세션 다시 시작] → [모두 실행]")
        print("=" * 70)
print(f"\n설치 단계 끝 ({(time.time() - t0) / 60:.1f}분). TRELLIS 사용 가능: {TRELLIS_READY}")


In [ ]:
# @title 작업 스크립트 준비 (자동)
SCRIPTS = {
    "gen_trellis.py": r'''# 로사리움 - TRELLIS 다중 이미지 -> 3D 생성 (Python 3.10 전용 환경에서 실행됨)
# 사용법: python gen_trellis.py config.json
# 종료 코드: 0 = 성공, 3 = GPU 메모리 부족(OOM), 그 밖 = 오류
import os
import sys
import json
import gc
import time
import traceback

os.environ.setdefault("ATTN_BACKEND", "xformers")         # T4 에는 flash-attn 이 없음
os.environ.setdefault("SPARSE_ATTN_BACKEND", "xformers")
os.environ.setdefault("SPCONV_ALGO", "native")            # 1회 실행에는 native 가 빠르고 안정적

cfg = json.load(open(sys.argv[1], encoding="utf-8"))
sys.path.insert(0, cfg["trellis_dir"])

import numpy as np
import torch
import imageio
from PIL import Image, ImageOps

OOM_EXIT = 3
WORK = cfg["work_dir"]
os.makedirs(WORK, exist_ok=True)


def is_oom(e):
    if isinstance(e, torch.cuda.OutOfMemoryError):
        return True
    s = str(e).lower()
    return "out of memory" in s or "cudaerrormemoryallocation" in s or "cuda_error_out_of_memory" in s


def ensure_model():
    """microsoft/TRELLIS-image-large (MIT) 를 고정 리비전으로 내려받는다. 토큰 불필요."""
    from huggingface_hub import snapshot_download
    d = cfg["model_dir"]
    snapshot_download(
        repo_id="microsoft/TRELLIS-image-large",
        revision=cfg["model_revision"],
        local_dir=d,
        allow_patterns=[
            "pipeline.json",
            "ckpts/ss_dec_conv3d_16l8_fp16.*",
            "ckpts/ss_flow_img_dit_L_16l8_fp16.*",
            "ckpts/slat_dec_gs_swin8_B_64l8gs32_fp16.*",
            "ckpts/slat_dec_mesh_swin8_B_64l8m256c_fp16.*",
            "ckpts/slat_flow_img_dit_L_64l8p2_fp16.*",
        ],
    )
    # 쓰지 않는 radiance-field 디코더는 빼서 다운로드/VRAM 을 아낀다.
    p = os.path.join(d, "pipeline.json")
    pj = json.load(open(p))
    if "slat_decoder_rf" in pj["args"]["models"]:
        del pj["args"]["models"]["slat_decoder_rf"]
        json.dump(pj, open(p, "w"), indent=2)
    return d


def load_image(path):
    im = Image.open(path)
    im = ImageOps.exif_transpose(im)
    has_alpha = im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info)
    if has_alpha:
        im = im.convert("RGBA")
        if np.array(im)[:, :, 3].min() == 255:  # 알파가 전부 불투명이면 배경 제거 필요
            im = im.convert("RGB")
    else:
        im = im.convert("RGB")
    return im


def sheet(images, size=256):
    ims = [i.convert("RGB").resize((size, size)) for i in images]
    out = Image.new("RGB", (size * len(ims), size), (0, 0, 0))
    for k, i in enumerate(ims):
        out.paste(i, (k * size, 0))
    return out


def main():
    t0 = time.time()
    from trellis.pipelines import TrellisImageTo3DPipeline
    from trellis.utils import render_utils

    print("[1/5] 모델 확인/다운로드 (처음 한 번만 약 3GB)...", flush=True)
    model_dir = ensure_model()

    print("[2/5] 모델 불러오는 중...", flush=True)
    pipe = TrellisImageTo3DPipeline.from_pretrained(model_dir)
    pipe.cuda()
    if cfg.get("rembg_model", "u2net") != "u2net":
        import rembg
        pipe.rembg_session = rembg.new_session(cfg["rembg_model"])
    print("   GPU 메모리 사용: %.1f GB" % (torch.cuda.memory_allocated() / 2**30), flush=True)

    print("[3/5] 배경 제거 + 정사각형 크롭 (첫 번째 = 정면)...", flush=True)
    processed = []
    for k, p in enumerate(cfg["images"]):
        out = pipe.preprocess_image(load_image(p))
        out.save(os.path.join(WORK, "pre_%02d.png" % k))
        processed.append(out)
        print("   %d. %s" % (k + 1, os.path.basename(p)), flush=True)
    sheet(processed).save(os.path.join(WORK, "preprocessed.png"))

    print("[4/5] 3D 생성 중 (%d장, mode=%s, steps=%d, seed=%d)..." % (
        len(processed), cfg["mode"], cfg["steps"], cfg["seed"]), flush=True)
    ss = {"steps": cfg["steps"], "cfg_strength": cfg["ss_guidance"]}
    sl = {"steps": cfg["steps"], "cfg_strength": cfg["slat_guidance"]}
    if len(processed) == 1:
        outputs = pipe.run(processed[0], seed=cfg["seed"], formats=["gaussian", "mesh"],
                           preprocess_image=False,
                           sparse_structure_sampler_params=ss, slat_sampler_params=sl)
    else:
        outputs = pipe.run_multi_image(processed, seed=cfg["seed"], formats=["gaussian", "mesh"],
                                       preprocess_image=False, mode=cfg["mode"],
                                       sparse_structure_sampler_params=ss, slat_sampler_params=sl)
    gs = outputs["gaussian"][0]
    mesh = outputs["mesh"][0]
    print("   원본 메시: 정점 %d개, 삼각형 %d개" % (mesh.vertices.shape[0], mesh.faces.shape[0]), flush=True)

    # 생성 모델은 더 이상 필요 없으므로 GPU 에서 내린다 (미리보기/저장용 메모리 확보).
    for m in pipe.models.values():
        m.cpu()
    del pipe, outputs
    gc.collect()
    torch.cuda.empty_cache()

    # 다음 단계(GLB 변환)를 별도 프로세스에서 하기 위해 결과를 저장한다.
    state = {
        "gaussian": dict(gs.init_params),
        "mesh": {"vertices": mesh.vertices.detach().cpu(), "faces": mesh.faces.detach().cpu()},
    }
    for k in ["_xyz", "_features_dc", "_features_rest", "_scaling", "_rotation", "_opacity"]:
        v = getattr(gs, k, None)
        state["gaussian"][k] = None if v is None else v.detach().cpu()
    torch.save(state, os.path.join(WORK, "state.pt"))

    print("[5/5] 미리보기 렌더링...", flush=True)
    bg = (0.12, 0.12, 0.14)
    yaws = [0.0, np.pi / 2, np.pi, 3 * np.pi / 2]
    extr, intr = render_utils.yaw_pitch_r_fov_to_extrinsics_intrinsics(yaws, [0.15] * 4, 2, 40)
    col = render_utils.render_frames(gs, extr, intr, {"resolution": 384, "bg_color": bg}, verbose=False)["color"]
    nrm = render_utils.render_frames(mesh, extr, intr, {"resolution": 384}, verbose=False)["normal"]
    grid = np.concatenate([np.concatenate(col, axis=1), np.concatenate(nrm, axis=1)], axis=0)
    Image.fromarray(grid).save(os.path.join(WORK, "preview_views.png"))
    v_col = render_utils.render_video(gs, resolution=384, num_frames=60, bg_color=bg, verbose=False)["color"]
    v_nrm = render_utils.render_video(mesh, resolution=384, num_frames=60, verbose=False)["normal"]
    frames = [np.concatenate([a, b], axis=1) for a, b in zip(v_col, v_nrm)]
    imageio.mimsave(os.path.join(WORK, "preview_turntable.mp4"), frames, fps=15)

    info = {
        "images": cfg["images"],
        "raw_faces": int(mesh.faces.shape[0]),
        "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
        "seconds": round(time.time() - t0, 1),
    }
    json.dump(info, open(os.path.join(WORK, "gen_info.json"), "w"), indent=2, ensure_ascii=False)
    print("완료: %.0f초, 최대 GPU 메모리 %.1f GB" % (info["seconds"], info["peak_vram_gb"]), flush=True)


if __name__ == "__main__":
    try:
        main()
    except Exception as e:  # noqa
        if is_oom(e):
            print("\n[GPU 메모리 부족] T4 GPU 메모리가 모자랍니다.", flush=True)
            sys.exit(OOM_EXIT)
        traceback.print_exc()
        sys.exit(1)
''',
    "export_glb.py": r'''# 로사리움 - TRELLIS 결과(state.pt) -> 텍스처 GLB 2종 (기본 + 모바일 경량)
# 사용법: python export_glb.py config.json
# 종료 코드: 0 = 성공(기본 GLB 생성), 3 = GPU 메모리 부족, 그 밖 = 오류
import os
import sys
import io
import json
import gc
import time
import traceback

os.environ.setdefault("ATTN_BACKEND", "xformers")
os.environ.setdefault("SPARSE_ATTN_BACKEND", "xformers")
os.environ.setdefault("SPCONV_ALGO", "native")

cfg = json.load(open(sys.argv[1], encoding="utf-8"))
sys.path.insert(0, cfg["trellis_dir"])

import numpy as np
import torch
from PIL import Image
from easydict import EasyDict as edict

OOM_EXIT = 3
WORK = cfg["work_dir"]
OUT = cfg["out_dir"]
os.makedirs(OUT, exist_ok=True)


def is_oom(e):
    if isinstance(e, torch.cuda.OutOfMemoryError):
        return True
    s = str(e).lower()
    return "out of memory" in s or "cudaerrormemoryallocation" in s


def load_state():
    from trellis.representations import Gaussian
    st = torch.load(os.path.join(WORK, "state.pt"), map_location="cpu")
    g = st["gaussian"]
    gs = Gaussian(
        aabb=g["aabb"], sh_degree=g["sh_degree"], mininum_kernel_size=g["mininum_kernel_size"],
        scaling_bias=g["scaling_bias"], opacity_bias=g["opacity_bias"],
        scaling_activation=g["scaling_activation"],
    )
    for k in ["_xyz", "_features_dc", "_features_rest", "_scaling", "_rotation", "_opacity"]:
        v = g.get(k)
        setattr(gs, k, None if v is None else v.cuda())
    mesh = edict(vertices=st["mesh"]["vertices"].cuda(), faces=st["mesh"]["faces"].cuda())
    return gs, mesh


def finish_material(tm, jpeg):
    """TRELLIS 기본 재질은 metallicFactor 가 빠져 있어 glTF 기본값(1.0=금속)이 된다.
    three.js 에서 어둡게 보이지 않도록 대리석에 맞게 비금속/반광택으로 지정한다."""
    mat = tm.visual.material
    mat.name = "marble"
    mat.metallicFactor = 0.0
    mat.roughnessFactor = 0.5
    if jpeg and getattr(mat, "baseColorTexture", None) is not None:
        buf = io.BytesIO()
        mat.baseColorTexture.convert("RGB").save(buf, format="JPEG", quality=90)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        mat.baseColorTexture = img  # format == JPEG -> GLB 안에 image/jpeg 로 저장됨
    return tm


def export(tm, path, jpeg):
    finish_material(tm, jpeg)
    tm.export(path, file_type="glb", include_normals=True)
    return len(tm.faces), os.path.getsize(path)


def main():
    t0 = time.time()
    from trellis.utils import postprocessing_utils
    gs, mesh = load_state()
    raw = int(mesh.faces.shape[0])
    print("원본 메시 삼각형: %d" % raw, flush=True)
    result = {"raw_faces": raw}

    # 1) 기본 GLB
    main_path = os.path.join(OUT, cfg["main_name"])
    print("[1/2] 기본 GLB (simplify=%.3f, 텍스처 %d) 만드는 중... (수 분 소요)" % (
        cfg["simplify"], cfg["texture_size"]), flush=True)
    tm = postprocessing_utils.to_glb(gs, mesh, simplify=cfg["simplify"],
                                     texture_size=cfg["texture_size"], verbose=True)
    n, size = export(tm, main_path, jpeg=False)
    result["main"] = {"path": main_path, "faces": n, "bytes": size}
    print("   -> %s : 삼각형 %d개, %.2f MB" % (os.path.basename(main_path), n, size / 1e6), flush=True)
    del tm
    gc.collect()
    torch.cuda.empty_cache()
    json.dump(result, open(os.path.join(WORK, "export_info.json"), "w"), indent=2, ensure_ascii=False)

    # 2) 모바일용 경량 GLB: 같은 결과를 더 강하게 단순화하고 텍스처를 다시 굽는다(UV 가 깨지지 않음).
    target = int(cfg["mobile_target_faces"])
    mob_path = os.path.join(OUT, cfg["mobile_name"])
    try:
        s = min(0.995, max(cfg["simplify"], 1.0 - target * 1.05 / max(raw, 1)))
        for attempt in range(2):
            print("[2/2] 모바일 GLB (목표 약 %d 삼각형, simplify=%.4f, 텍스처 %d)..." % (
                target, s, cfg["mobile_texture_size"]), flush=True)
            tm = postprocessing_utils.to_glb(gs, mesh, simplify=s,
                                             texture_size=cfg["mobile_texture_size"], verbose=True)
            n = len(tm.faces)
            if attempt == 0 and n > target * 1.35 and s < 0.995:
                # 비율을 실제 결과에 맞춰 한 번만 다시 계산
                eff_raw = n / max(1e-6, (1.0 - s))
                s = min(0.995, 1.0 - target / eff_raw)
                print("   삼각형 %d개 -> 목표보다 많아 한 번 더 줄입니다." % n, flush=True)
                del tm
                gc.collect()
                torch.cuda.empty_cache()
                continue
            n, size = export(tm, mob_path, jpeg=True)
            result["mobile"] = {"path": mob_path, "faces": n, "bytes": size}
            print("   -> %s : 삼각형 %d개, %.2f MB" % (os.path.basename(mob_path), n, size / 1e6), flush=True)
            break

    except Exception as e:  # 모바일 GLB 실패는 치명적이지 않음 (기본 GLB 는 이미 저장됨)
        if is_oom(e):
            print("   [경고] 모바일 GLB 를 만들 GPU 메모리가 부족합니다. 기본 GLB 만 사용하세요.", flush=True)
        else:
            traceback.print_exc()
            print("   [경고] 모바일 GLB 생성 실패. 기본 GLB 는 정상입니다.", flush=True)
        torch.cuda.empty_cache()

    result["seconds"] = round(time.time() - t0, 1)
    result["peak_vram_gb"] = round(torch.cuda.max_memory_allocated() / 2**30, 2)
    json.dump(result, open(os.path.join(WORK, "export_info.json"), "w"), indent=2, ensure_ascii=False)
    print("완료: %.0f초, 최대 GPU 메모리 %.1f GB" % (result["seconds"], result["peak_vram_gb"]), flush=True)


if __name__ == "__main__":
    try:
        main()
    except Exception as e:  # noqa
        if is_oom(e):
            print("\n[GPU 메모리 부족] GLB 변환 중 메모리가 모자랍니다.", flush=True)
            sys.exit(OOM_EXIT)
        traceback.print_exc()
        sys.exit(1)
''',
    "gen_triposr.py": r'''# 로사리움 - 대체 경로: TripoSR (MIT) 로 정면 사진 1장 -> 정점 색상 GLB
# 사용법: python gen_triposr.py config.json
import os
import sys
import json
import time
import struct
import traceback

cfg = json.load(open(sys.argv[1], encoding="utf-8"))
sys.path.insert(0, cfg["triposr_dir"])
sys.path.insert(0, cfg["stub_dir"])  # torchmcubes 대신 scikit-image 마칭큐브 사용 (컴파일 불필요)

import numpy as np
import torch
import trimesh
import pyvista as pv
from PIL import Image, ImageOps

OUT = cfg["out_dir"]
os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def patch_glb_material(data, metallic=0.0, roughness=0.6):
    """정점 색상 GLB 에는 재질이 없어 three.js 기본값(metalness=1)이 쓰인다. 비금속 재질을 넣는다."""
    magic, ver, _ = struct.unpack_from("<III", data, 0)
    jlen, jtype = struct.unpack_from("<II", data, 12)
    j = json.loads(data[20:20 + jlen])
    rest = data[20 + jlen:]
    mats = j.setdefault("materials", [])
    if not mats:
        mats.append({"name": "marble", "pbrMetallicRoughness": {"baseColorFactor": [1, 1, 1, 1]}})
    for mat in mats:
        pbr = mat.setdefault("pbrMetallicRoughness", {})
        pbr["metallicFactor"] = metallic
        pbr["roughnessFactor"] = roughness
    for m in j.get("meshes", []):
        for prim in m["primitives"]:
            prim.setdefault("material", 0)
    js = json.dumps(j, separators=(",", ":")).encode("utf-8")
    js += b" " * ((4 - len(js) % 4) % 4)
    return (struct.pack("<III", magic, ver, 20 + len(js) + len(rest))
            + struct.pack("<II", len(js), jtype) + js + rest)


def decimate(vertices, faces, target):
    if len(faces) <= target:
        return vertices, faces
    pd = pv.PolyData(vertices, np.hstack([np.full((len(faces), 1), 3), faces]).astype(np.int64))
    pd = pd.decimate(1.0 - target / float(len(faces)))
    return np.asarray(pd.points), pd.faces.reshape(-1, 4)[:, 1:]


def main():
    t0 = time.time()
    import rembg
    from huggingface_hub import snapshot_download
    from tsr.system import TSR
    from tsr.utils import remove_background, resize_foreground, to_gradio_3d_orientation

    print("[1/4] TripoSR 모델 준비...", flush=True)
    mdir = snapshot_download("stabilityai/TripoSR", revision=cfg["model_revision"],
                             local_dir=cfg["model_dir"], allow_patterns=["config.yaml", "model.ckpt"])
    model = TSR.from_pretrained(mdir, config_name="config.yaml", weight_name="model.ckpt")
    model.renderer.set_chunk_size(8192)
    model.to(DEV)

    print("[2/4] 정면 이미지 배경 제거: %s" % os.path.basename(cfg["image"]), flush=True)
    img = ImageOps.exif_transpose(Image.open(cfg["image"])).convert("RGB")
    s = min(1.0, 1024.0 / max(img.size))
    if s < 1:
        img = img.resize((int(img.width * s), int(img.height * s)), Image.LANCZOS)
    img = remove_background(img, rembg.new_session(cfg.get("rembg_model", "u2net")))
    img = resize_foreground(img, 0.85)
    arr = np.array(img).astype(np.float32) / 255.0
    arr = arr[:, :, :3] * arr[:, :, 3:4] + (1 - arr[:, :, 3:4]) * 0.5
    img = Image.fromarray((arr * 255.0).astype(np.uint8))
    img.save(os.path.join(cfg["work_dir"], "triposr_input.png"))

    print("[3/4] 3D 생성 + 메시 추출...", flush=True)
    with torch.no_grad():
        scene_codes = model([img], device=DEV)
    mesh = model.extract_mesh(scene_codes, False, resolution=cfg.get("mc_resolution", 256))[0]
    v0, f0 = np.asarray(mesh.vertices), np.asarray(mesh.faces)
    print("   원본 삼각형 %d개" % len(f0), flush=True)

    print("[4/4] 단순화 + 정점 색상 + GLB 저장...", flush=True)
    result = {}
    for key, target, name in [("main", cfg["main_faces"], cfg["main_name"]),
                              ("mobile", cfg["mobile_faces"], cfg["mobile_name"])]:
        v, f = decimate(v0, f0, int(target))
        with torch.no_grad():
            col = model.renderer.query_triplane(
                model.decoder, torch.tensor(v, dtype=torch.float32, device=DEV), scene_codes[0])["color"]
        col = (np.clip(col.float().cpu().numpy(), 0, 1) * 255).astype(np.uint8)
        col = np.concatenate([col, np.full((len(col), 1), 255, np.uint8)], axis=1)
        tm = trimesh.Trimesh(vertices=v, faces=f, vertex_colors=col, process=False)
        tm = to_gradio_3d_orientation(tm)  # TripoSR 데모와 같은 Y-up 방향
        if tm.volume < 0:
            tm.invert()  # 면 방향(법선)이 뒤집혔으면 바로잡기
        data = patch_glb_material(tm.export(file_type="glb", include_normals=True))
        path = os.path.join(OUT, name)
        with open(path, "wb") as fh:
            fh.write(data)
        result[key] = {"path": path, "faces": int(len(tm.faces)), "bytes": len(data)}
        print("   -> %s : 삼각형 %d개, %.2f MB" % (name, len(tm.faces), len(data) / 1e6), flush=True)
    result["seconds"] = round(time.time() - t0, 1)
    json.dump(result, open(os.path.join(cfg["work_dir"], "triposr_info.json"), "w"), indent=2)
    print("완료: %.0f초" % result["seconds"], flush=True)


if __name__ == "__main__":
    try:
        main()
    except Exception:
        traceback.print_exc()
        sys.exit(1)
''',
    "torchmcubes.py": r'''# torchmcubes 대체 모듈 (TripoSR 용). CUDA 컴파일 없이 scikit-image 로 마칭큐브를 수행한다.
# torchmcubes.marching_cubes 와 같은 규약: 정점 좌표를 (k, j, i) 순서로 돌려준다.
import numpy as np
import torch
from skimage.measure import marching_cubes as _mc


def marching_cubes(vol, thresh):
    v = vol.detach().float().cpu().numpy()
    verts, faces, _, _ = _mc(v, level=float(thresh))
    verts = np.ascontiguousarray(verts[:, ::-1])
    faces = np.ascontiguousarray(faces[:, ::-1])  # 축 순서를 바꿨으므로 면 방향도 뒤집어 법선이 바깥을 향하게
    return torch.from_numpy(verts).float(), torch.from_numpy(faces.astype(np.int64))
''',
}
os.makedirs(f"{SCRIPT_DIR}/stub", exist_ok=True)
for _n, _s in SCRIPTS.items():
    _p = f"{SCRIPT_DIR}/stub/{_n}" if _n == "torchmcubes.py" else f"{SCRIPT_DIR}/{_n}"
    open(_p, "w", encoding="utf-8").write(_s)
print("스크립트 준비 완료:", ", ".join(SCRIPTS))


## 4. 3D 생성 (TRELLIS, 여러 장 동시 사용)
배경 제거 결과와, 생성된 모델을 네 방향에서 본 미리보기(위: 색, 아래: 형태)와 회전 영상이 나옵니다.


In [ ]:
# @title 3D 생성 + 미리보기
import base64
from IPython.display import display, HTML, Image as IPyImage

TRELLIS_OK = False
if "INPUT_IMAGES" not in globals():
    INPUT_IMAGES = json.load(open(f"{ROOT}/inputs.json"))
if "TRELLIS_READY" not in globals():
    TRELLIS_READY = os.path.exists(f"{ROOT}/.ok_trellis")


def trellis_config(images):
    cfg = dict(images=images, seed=int(SEED), steps=int(STEPS), ss_guidance=float(SS_GUIDANCE),
               slat_guidance=float(SLAT_GUIDANCE), mode=MULTI_MODE, rembg_model=REMBG_MODEL,
               work_dir=WORK_DIR, out_dir=OUT_DIR, trellis_dir=f"{SRC_DIR}/TRELLIS",
               model_dir=f"{CACHE_DIR}/TRELLIS-image-large", model_revision=TRELLIS_MODEL_REV,
               simplify=float(SIMPLIFY), texture_size=int(TEXTURE_SIZE),
               mobile_target_faces=int(MOBILE_TARGET_FACES), mobile_texture_size=int(MOBILE_TEXTURE_SIZE),
               main_name=f"{OUTPUT_BASENAME}_trellis.glb", mobile_name=f"{OUTPUT_BASENAME}_trellis_mobile.glb")
    path = f"{WORK_DIR}/trellis_config.json"
    json.dump(cfg, open(path, "w"), ensure_ascii=False, indent=2)
    return path


def show_video(path, width=768):
    data = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop muted playsinline>'
                 f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))


OOM_MSG = """
[GPU 메모리 부족]
 - 사진 수를 줄이면(예: 정면/옆/뒤 3장) 메모리를 덜 씁니다.
 - [런타임] → [세션 다시 시작] 후 [모두 실행] 하면 메모리가 깨끗한 상태에서 다시 시도합니다.
 - 그래도 안 되면 아래 대체 방법(TripoSR)이 자동으로 실행됩니다.
"""

if FORCE_TRIPOSR or not TRELLIS_READY:
    print("TRELLIS 를 사용할 수 없어 이 단계를 건너뜁니다. 아래 대체 방법(TripoSR)으로 진행합니다.")
else:
    for _f in ["state.pt", "gen_info.json"]:
        if os.path.exists(f"{WORK_DIR}/{_f}"):
            os.remove(f"{WORK_DIR}/{_f}")
    imgs = list(INPUT_IMAGES)
    rc = run_stream([VENV_PY, "-u", f"{SCRIPT_DIR}/gen_trellis.py", trellis_config(imgs)], "generate.log",
                    cwd=f"{SRC_DIR}/TRELLIS")
    if rc == 3 and len(imgs) > 3:
        print(OOM_MSG)
        # 정면 + 옆 + 뒤 (파일 이름으로 찾고, 없으면 고르게 고름) 3장으로 한 번 더
        def _pick(word, default):
            c = [p for p in imgs[1:] if word in os.path.basename(p).lower() and "quarter" not in os.path.basename(p).lower()]
            return c[0] if c else default
        _side, _back = _pick("side", imgs[len(imgs) // 2]), _pick("back", imgs[-1])
        imgs = [imgs[0]] + [p for p in dict.fromkeys([_side, _back]) if p != imgs[0]]
        print("사진 3장으로 다시 시도합니다:", [os.path.basename(p) for p in imgs])
        rc = run_stream([VENV_PY, "-u", f"{SCRIPT_DIR}/gen_trellis.py", trellis_config(imgs)], "generate.log",
                        cwd=f"{SRC_DIR}/TRELLIS")
    if rc == 0 and os.path.exists(f"{WORK_DIR}/state.pt"):
        TRELLIS_OK = True
        print("\n배경 제거 + 크롭 결과 (모델에 들어간 그림):")
        display(IPyImage(filename=f"{WORK_DIR}/preprocessed.png"))
        print("생성 결과 미리보기 (위: 색/질감, 아래: 형태):")
        display(IPyImage(filename=f"{WORK_DIR}/preview_views.png"))
        show_video(f"{WORK_DIR}/preview_turntable.mp4")
        print("마음에 들지 않으면 0번 설정에서 SEED 를 바꾸고 이 칸부터 다시 실행하세요 ([런타임] → [이후 셀 실행]).")
    elif rc == 3:
        print(OOM_MSG)
    else:
        print("\nTRELLIS 생성에 실패했습니다. 위 오류 메시지를 확인하세요. 아래 대체 방법으로 진행합니다.")
        print("(로그: /content/rosarium/logs/generate.log)")


In [ ]:
# @title GLB 만들기 (기본 + 모바일 경량)
TRELLIS_GLBS = []
_cfgp = f"{WORK_DIR}/trellis_config.json"
_have_state = os.path.exists(f"{WORK_DIR}/state.pt") and os.path.exists(_cfgp)
if _have_state and not FORCE_TRIPOSR and globals().get("TRELLIS_OK", True):
    # 설정 칸에서 바꾼 SIMPLIFY / 텍스처 크기 등을 반영 (이 칸만 다시 실행해도 됨)
    _c = json.load(open(_cfgp))
    _c.update(simplify=float(SIMPLIFY), texture_size=int(TEXTURE_SIZE), out_dir=OUT_DIR,
              mobile_target_faces=int(MOBILE_TARGET_FACES), mobile_texture_size=int(MOBILE_TEXTURE_SIZE))
    json.dump(_c, open(_cfgp, "w"), ensure_ascii=False, indent=2)
    if os.path.exists(f"{WORK_DIR}/export_info.json"):
        os.remove(f"{WORK_DIR}/export_info.json")
    TRELLIS_OK = True
    rc = run_stream([VENV_PY, "-u", f"{SCRIPT_DIR}/export_glb.py", f"{WORK_DIR}/trellis_config.json"],
                    "export.log", cwd=f"{SRC_DIR}/TRELLIS")
    info = json.load(open(f"{WORK_DIR}/export_info.json")) if os.path.exists(f"{WORK_DIR}/export_info.json") else {}
    for key in ["main", "mobile"]:
        if key in info and os.path.exists(info[key]["path"]):
            TRELLIS_GLBS.append(info[key]["path"])
    if TRELLIS_GLBS:
        print("\n결과 파일:")
        for key, label in [("main", "기본"), ("mobile", "모바일")]:
            if key in info:
                print(f"  [{label}] {os.path.basename(info[key]['path'])} : 삼각형 {info[key]['faces']:,}개, "
                      f"{info[key]['bytes'] / 1e6:.2f} MB")
        if "main" in info and info["main"]["faces"] < 20000:
            print("  (삼각형이 2만 개보다 적습니다. 더 정교하게 하려면 SIMPLIFY 를 0.9 로 낮추고 이 칸만 다시 실행하세요.)")
    else:
        TRELLIS_OK = False
        if rc == 3:
            print("\n[GPU 메모리 부족] GLB 변환 실패. [런타임] → [세션 다시 시작] 후 이 칸만 다시 실행해 보세요.")
            print("(생성 결과는 저장되어 있어 '설정', 'GPU 확인', '작업 스크립트 준비', 이 칸만 실행하면 됩니다.)")
        print("GLB 를 만들지 못했습니다. 아래 대체 방법(TripoSR)으로 진행합니다. (로그: /content/rosarium/logs/export.log)")
else:
    print("TRELLIS 결과가 없어 건너뜁니다.")


## 5. 대체 방법: TripoSR (TRELLIS 가 실패했을 때만 자동 실행)
정면 사진 **1장**만 사용하므로 **뒷면은 추측으로 만들어져 덜 정확**합니다. 색은 정점 색상으로 들어갑니다.


In [ ]:
# @title TripoSR 대체 생성 (TRELLIS 성공 시 건너뜀)
FALLBACK_GLBS = []
if globals().get("TRELLIS_OK") and globals().get("TRELLIS_GLBS"):
    print("TRELLIS 결과가 있으므로 대체 방법은 건너뜁니다.")
else:
    if "INPUT_IMAGES" not in globals():
        INPUT_IMAGES = json.load(open(f"{ROOT}/inputs.json"))
    print("TripoSR 준비 중 (정면 사진 1장 사용, 뒷면은 덜 정확합니다)...")
    _env = clean_env()
    _uv = globals().get("UV_BIN") or find_uv()
    _cmd = f"""set -eo pipefail
D={SRC_DIR}/TripoSR
if [ "$(git -C $D rev-parse HEAD 2>/dev/null)" != "107cefdc244c39106fa830359024f6a2f1c78871" ]; then
  rm -rf $D; mkdir -p $D; git -C $D init -q; git -C $D remote add origin https://github.com/VAST-AI-Research/TripoSR.git
  git -C $D fetch -q --depth 1 origin 107cefdc244c39106fa830359024f6a2f1c78871; git -C $D -c advice.detachedHead=false checkout -q FETCH_HEAD
fi
{_uv} pip install --python {VENV_PY} "omegaconf==2.3.0" "einops==0.7.0" "transformers==4.46.3"
"""
    rc = run_stream(["bash", "-c", _cmd], "triposr_install.log", env=_env)
    if rc == 0:
        _cfg = dict(image=INPUT_IMAGES[0], triposr_dir=f"{SRC_DIR}/TripoSR", stub_dir=f"{SCRIPT_DIR}/stub",
                    work_dir=WORK_DIR, out_dir=OUT_DIR, model_dir=f"{CACHE_DIR}/TripoSR",
                    model_revision=TRIPOSR_MODEL_REV, rembg_model=REMBG_MODEL, mc_resolution=256,
                    main_faces=40000, mobile_faces=int(MOBILE_TARGET_FACES),
                    main_name=f"{OUTPUT_BASENAME}_triposr.glb", mobile_name=f"{OUTPUT_BASENAME}_triposr_mobile.glb")
        json.dump(_cfg, open(f"{WORK_DIR}/triposr_config.json", "w"), ensure_ascii=False, indent=2)
        rc = run_stream([VENV_PY, "-u", f"{SCRIPT_DIR}/gen_triposr.py", f"{WORK_DIR}/triposr_config.json"],
                        "triposr.log", env=_env)
    if rc == 0:
        info = json.load(open(f"{WORK_DIR}/triposr_info.json"))
        FALLBACK_GLBS = [info[k]["path"] for k in ["main", "mobile"] if k in info]
        print("\n대체 결과 (TripoSR, 정면 1장 기반 - 뒷면은 덜 정확):")
        for p in FALLBACK_GLBS:
            print(f"  {os.path.basename(p)}  {os.path.getsize(p) / 1e6:.2f} MB")
    else:
        print("\n대체 방법도 실패했습니다. 맨 아래 '문제 해결' 을 참고해 [런타임] → [세션 다시 시작] → [모두 실행] 해 보세요.")
        print("(로그: /content/rosarium/logs/)")


## 6. 다운로드
GLB 파일이 자동으로 내려받아집니다. 브라우저가 **"여러 파일 다운로드 허용"** 을 물으면 **허용**을 누르세요.
자동 다운로드가 안 되면 왼쪽 **폴더 아이콘** → `rosarium/output` 폴더에서 파일 옆 **⋮ → 다운로드** 를 누르세요.
받은 GLB 파일(특히 `_mobile.glb`)을 채팅에 올려 주시면 앱에 넣어 드립니다.


In [ ]:
# @title 결과 다운로드
_all = [p for p in globals().get("TRELLIS_GLBS", []) + globals().get("FALLBACK_GLBS", []) if os.path.exists(p)]
if not _all:
    _all = sorted(os.path.join(OUT_DIR, f) for f in os.listdir(OUT_DIR) if f.endswith(".glb"))
if not _all:
    print("만들어진 GLB 파일이 없습니다. 위쪽 칸들의 오류 메시지와 '문제 해결' 을 확인하세요.")
else:
    for p in _all:
        print(f"  {os.path.basename(p)}  ({os.path.getsize(p) / 1e6:.2f} MB)")
    if AUTO_DOWNLOAD and IN_COLAB:
        from google.colab import files
        for p in _all:
            files.download(p)
            time.sleep(1.5)
        print("다운로드를 시작했습니다. (브라우저의 다운로드 목록을 확인하세요)")


In [ ]:
# @title (선택) Google Drive 에 저장 - 필요하면 아래 줄들의 맨 앞 # 를 지우고 실행
# from google.colab import drive
# drive.mount("/content/drive")
# _dst = "/content/drive/MyDrive/rosarium_statue3d"
# os.makedirs(_dst, exist_ok=True)
# for _f in os.listdir(OUT_DIR):
#     shutil.copy(os.path.join(OUT_DIR, _f), _dst)
# for _f in ["preview_views.png", "preview_turntable.mp4", "preprocessed.png"]:
#     if os.path.exists(os.path.join(WORK_DIR, _f)):
#         shutil.copy(os.path.join(WORK_DIR, _f), _dst)
# print("Drive 저장 완료:", _dst)


## 문제 해결

**GPU 사용량 한도 / "GPU 백엔드에 연결할 수 없습니다"**
무료 Colab 은 하루 GPU 사용 시간이 정해져 있습니다. 몇 시간~하루 뒤에 다시 하세요.
(한 번 실행에 보통 20~35분이 걸리므로, 같은 날 여러 번 돌리면 한도에 걸리기 쉽습니다.)

**설치 중 빨간 오류**
1. 메뉴 **런타임 → 세션 다시 시작** → **런타임 → 모두 실행** (사진은 다시 올려야 합니다)
2. 그래도 같으면 **런타임 → 런타임 연결 해제 및 삭제** 후 처음부터 (새 컴퓨터를 받습니다)
3. 그래도 안 되면 `/content/rosarium/logs/` 폴더의 `install_trellis.log` 마지막 부분을 채팅에 붙여 주세요.

**GPU 메모리 부족 (OOM)**
- 자동으로 사진 3장(정면/중간/마지막)으로 한 번 더 시도합니다.
- 직접 줄이려면 정면·옆·뒤 3장만 올려 보세요.
- **런타임 → 세션 다시 시작** 후 **모두 실행** 하면 메모리가 비워진 상태로 다시 시작합니다.
- 그래도 안 되면 대체 방법(TripoSR)이 자동으로 실행됩니다. `FORCE_TRIPOSR` 를 켜면 처음부터 TripoSR 만 씁니다.

**결과 모양이 이상할 때**
- 0번 설정의 `SEED` 를 바꿔(예: 2, 3, 7) **런타임 → 이후 셀 실행** 을 4번 칸에서 누르세요. (설치는 다시 하지 않습니다)
- 배경 제거가 조각상 일부를 잘라 먹었다면(미리보기 첫 줄 확인) `REMBG_MODEL` 을 `birefnet-general-lite` 로 바꿔 보세요.
- `MULTI_MODE` 를 `multidiffusion` 으로 바꿔 볼 수 있지만, 몸이 뚱뚱/홀쭉해지는 왜곡이 보고된 방식이라 기본은 `stochastic` 입니다.

**삼각형 수 조절**
`SIMPLIFY` (기본 0.95) 를 0.9 로 낮추면 더 정교(삼각형 증가), 0.97 로 높이면 더 가벼워집니다. 생성 결과는 저장되어 있으므로 **GLB 만들기** 칸만 다시 실행하면 됩니다.

**앱에서 방향이 다를 때**
GLB 는 Y 축이 위쪽(glTF 표준)입니다. 정면 방향이 앱과 다르면 앱 쪽에서 Y 축 회전으로 맞춥니다.

---
사용 소프트웨어: TRELLIS (MIT, Microsoft), TripoSR (MIT, Tripo AI / Stability AI), DINOv2 (Apache-2.0), rembg/U²-Net (MIT/Apache-2.0),
nvdiffrast (NVIDIA Source Code License), mip-splatting diff-gaussian-rasterization (Inria, 비상업·연구용 라이선스 — 텍스처 굽기에만 사용되며 앱에는 포함되지 않음).
